In [ ]:
%%bash
echo "Installing docker..."
sudo pacman -S --noconfirm --needed docker >/dev/null


### 配置 Docker


In [ ]:
%%bash
daemon_file="/etc/docker/daemon.json"

sudo systemctl stop docker &>/dev/null

sudo mkdir -p /etc/docker
sudo tee "$daemon_file" > /dev/null <<'EOF'
{
  "registry-mirrors": [
    "https://docker.1ms.run",
    "https://docker.1panel.live",
    "https://docker.m.ixdev.cn",
    "https://hub.rat.dev",
    "https://dockerproxy.net",
    "https://docker.hlmirror.com",
    "https://hub1.nat.tf",
    "https://hub3.nat.tf",
    "https://docker.m.daocloud.io",
    "https://docker.kejilion.pro",
    "https://hub.1panel.dev",
    "https://dockerproxy.cool",
    "https://proxy.vvvv.ee"
  ]
}
EOF

sudo usermod -aG docker "$USER" &>/dev/null

sudo systemctl daemon-reload
sudo systemctl enable --now docker &>/dev/null

echo "✓ Docker configured"
echo "⚠ Please logout and login again for group changes to take effect"


# Docker 应用创建教程


## 1. Docker 基础概念

- **镜像(Image)**: 应用的只读模板
- **容器(Container)**: 镜像的运行实例
- **Dockerfile**: 构建镜像的脚本
- **仓库(Registry)**: 存储镜像的地方(如 Docker Hub)


## 2. 创建简单 Python Web 应用


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>步骤1: 创建项目目录>>>>>>>>>>>>>>>>
mkdir -p docker_demo
cd docker_demo


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>步骤2: 创建应用代码>>>>>>>>>>>>>>>>
cat > docker_demo/app.py << 'EOF'
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# >>>>>>>>>>>>>>>简单的 Flask Web 应用>>>>>>>>>>>>>>>>

from flask import Flask

app = Flask(__name__)

@app.route('/')
def hello():
    return "Hello from Docker!"

if __name__ == "__main__":
    app.run(host='0.0.0.0', port=5000)
EOF


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>步骤3: 创建依赖文件>>>>>>>>>>>>>>>>
cat > docker_demo/requirements.txt << 'EOF'
flask==3.0.0
EOF


## 3. 编写 Dockerfile


In [ ]:
%%bash
cat > docker_demo/Dockerfile << 'EOF'
# >>>>>>>>>>>>>>>>>>>>基础镜像>>>>>>>>>>>>>>>>
FROM python:3.11-slim

# >>>>>>>>>>>>>>>>>>>>设置工作目录>>>>>>>>>>>>>>>>
WORKDIR /app

# >>>>>>>>>>>>>>>>>>>>复制依赖文件>>>>>>>>>>>>>>>>
COPY requirements.txt .

# >>>>>>>>>>>>>>>>>>>>安装依赖>>>>>>>>>>>>>>>>
RUN pip install --no-cache-dir -r requirements.txt

# >>>>>>>>>>>>>>>>>>>>复制应用代码>>>>>>>>>>>>>>>>
COPY app.py .

# >>>>>>>>>>>>>>>>>>>>暴露端口>>>>>>>>>>>>>>>>
EXPOSE 5000

# >>>>>>>>>>>>>>>>>>>>启动命令>>>>>>>>>>>>>>>>
CMD ["python", "app.py"]
EOF


## 4. 构建镜像


In [ ]:
%%bash
cd docker_demo
docker build -t my-flask-app:latest .


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/



Sending build context to Docker daemon  4.608kB
Step 1/7 : FROM python:3.11-slim
3.11-slim: Pulling from library/python
119d43eec815: Pulling fs layer
0b2bf04f68e9: Pulling fs layer
3e731abb5c1d: Pulling fs layer
5b09819094bb: Pulling fs layer
5b09819094bb: Download complete
6a59d23a42b3: Download complete
5d468228012d: Download complete
3e731abb5c1d: Download complete
0b2bf04f68e9: Download complete
119d43eec815: Download complete
119d43eec815: Pull complete
5b09819094bb: Pull complete
3e731abb5c1d: Pull complete
0b2bf04f68e9: Pull complete
Digest: sha256:c24e9effa2821a6885165d930d939fec2af0dcf819276138f11dd45e200bd032
Status: Downloaded newer image for python:3.11-slim
 ---> c24e9effa282
Step 2/7 : WORKDIR /app
 ---> Running in 67271609cd3b
 ---> Removed intermediate container 67271609cd3b
 ---> 750f17740e22
Step 3/7 : COPY requirements.txt .
 ---> 3dcc3d216393
Step 4/7 : RUN pip install --no-cache-dir -r requirements.txt
 ---> Running in 50e22c09e4af
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5. 运行容器


In [ ]:
%%bash
docker run -d  \
  --name flask-demo \
  -p 5001:5000 \
  my-flask-app:latest


7c0361d879e119aec8dbd7b756bbcd3fc59549bd49901c6ef13ede4215f965ad


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>测试应用>>>>>>>>>>>>>>>>
curl http://localhost:5001


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    18 100    18   0     0 14007     0  --:--:-- --:--:-- --:--:-- 18000


Hello from Docker!

## 6. 带数据持久化的应用


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>使用卷挂载>>>>>>>>>>>>>>>>
docker run -d \
  --name app-with-data \
  -p 8080:5000 \
  -v /data/.docker/app-data:/app/data \
  my-flask-app:latest


## 7. 常用 Docker 命令


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>查看运行中的容器>>>>>>>>>>>>>>>>
docker ps


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>查看所有容器>>>>>>>>>>>>>>>>
docker ps -a


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>查看容器日志>>>>>>>>>>>>>>>>
docker logs flask-demo


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>停止容器>>>>>>>>>>>>>>>>
docker stop flask-demo


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>启动容器>>>>>>>>>>>>>>>>
docker start flask-demo


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>删除容器>>>>>>>>>>>>>>>>
docker rm flask-demo


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>查看镜像>>>>>>>>>>>>>>>>
docker images


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>删除镜像>>>>>>>>>>>>>>>>
docker rmi my-flask-app:latest


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>进入容器>>>>>>>>>>>>>>>>
docker exec -it flask-demo /bin/bash


## 8. Docker Compose 示例


In [ ]:
%%bash
cat > docker_demo/docker-compose.yml << 'EOF'
version: '3.8'

services:
  web:
    build: .
    ports:
      - "5000:5000"
    volumes:
      - /data/.docker/app-data:/app/data
    restart: unless-stopped
EOF


In [ ]:
%%bash
cd docker_demo
# >>>>>>>>>>>>>>>>>>>>启动服务>>>>>>>>>>>>>>>>
docker-compose up -d


In [ ]:
%%bash
cd docker_demo
# >>>>>>>>>>>>>>>>>>>>停止服务>>>>>>>>>>>>>>>>
docker-compose down


## 9. 实际应用示例


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>Memos 笔记应用>>>>>>>>>>>>>>>>
docker run -d \
  --name memos \
  -p 5230:5230 \
  -v /data/.home/.memos:/var/opt/memos \
  neosmemo/memos:stable


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>Nginx Web 服务器>>>>>>>>>>>>>>>>
docker run -d \
  --name nginx \
  -p 80:80 \
  -v /data/.docker/nginx/html:/usr/share/nginx/html \
  nginx:alpine


## 10. 清理资源


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>清理未使用的容器>>>>>>>>>>>>>>>>
docker container prune -f


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>清理未使用的镜像>>>>>>>>>>>>>>>>
docker image prune -a -f


In [ ]:
%%bash
# >>>>>>>>>>>>>>>>>>>>清理所有未使用资源>>>>>>>>>>>>>>>>
docker system prune -a -f --volumes
